<a href="https://colab.research.google.com/github/PujaSridhar/finsight/blob/main/Fine_tuning_Mistral_7B_Phase_1_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight — Phase 1: Data Preparation

**Goal:** Build a clean instruction-following dataset of 5K–10K examples for fine-tuning Mistral-7B on financial analysis tasks.

**Sources:**
- FinanceBench (HuggingFace) — Q&A pairs grounded in real 10-K/10-Q filings
- SEC EDGAR full-text search API — earnings call transcripts and filings

**Output format:**
```json
{"instruction": "...", "input": "...", "output": "..."}
```

**Estimated runtime:** ~20–30 minutes (mostly EDGAR API calls)

## 1. Install dependencies

In [ ]:
!pip install -q datasets pandas requests tqdm transformers
print('✓ Dependencies installed')

✓ Dependencies installed


In [ ]:
import json
import re
import time
import hashlib
import random
from pathlib import Path
from collections import defaultdict

import pandas as pd
import requests
from datasets import load_dataset
from tqdm.auto import tqdm

random.seed(42)

# ── Output paths ──────────────────────────────────────────
OUTPUT_DIR = Path('finsight_data')
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_DIR = OUTPUT_DIR / 'raw'
RAW_DIR.mkdir(exist_ok=True)

TRAIN_FILE  = OUTPUT_DIR / 'train.jsonl'
VAL_FILE    = OUTPUT_DIR / 'val.jsonl'
STATS_FILE  = OUTPUT_DIR / 'dataset_stats.json'

print('✓ Imports OK')
print(f'  Output dir: {OUTPUT_DIR.resolve()}')

✓ Imports OK
  Output dir: /content/finsight_data


## 2. Source A — FinanceBench

In [ ]:
# FinanceBench: ~10K Q&A pairs grounded in real 10-K / 10-Q / earnings filings.
# Each row has: question, answer, evidence (a passage), doc_name, doc_year.
# We'll convert each row into an instruction-following triple.

print('Loading FinanceBench from HuggingFace...')
fb_dataset = load_dataset('PatronusAI/financebench', split='train')
print(f'  Rows: {len(fb_dataset)}')
print(f'  Columns: {fb_dataset.column_names}')
fb_dataset[0]  # inspect one row

Loading FinanceBench from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

financebench_merged.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

  Rows: 150
  Columns: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']


{'financebench_id': 'financebench_id_03029',
 'company': '3M',
 'doc_name': '3M_2018_10K',
 'question_type': 'metrics-generated',
 'question_reasoning': 'Information extraction',
 'domain_question_num': None,
 'question': 'What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.',
 'answer': '$1577.00',
 'justification': 'The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).',
 'dataset_subset_label': 'OPEN_SOURCE',
 'evidence': [{'evidence_text': 'Table of Contents \n3M Company and Subsidiaries\nConsolidated Statement of Cash Flow s\nYears ended December 31\n \n(Millions)\n \n2018\n \n2017\n \n2016\n \nCash Flows from Operating Activities\n \n \n \n \n \n \n \nNet income including noncontrolling interest\n \n$\n5,363 \n$\n4,869 \n$\n5,058 \nAdjustments to reconci

In [ ]:
# ── Instruction templates for FinanceBench ────────────────
# We vary the instruction phrasing so the model doesn't overfit to one template.

TEMPLATES = [
    'Answer the following question based on the provided financial document excerpt.',
    'Using the financial document excerpt below, answer the question concisely and accurately.',
    'You are a financial analyst. Answer the question based strictly on the evidence provided.',
    'Read the excerpt from the financial filing and answer the question that follows.',
    'Based on the following passage from a financial document, provide a precise answer.',
]

def fb_row_to_example(row, template_pool=TEMPLATES):
    """Convert a FinanceBench row into an instruction-following dict."""
    # Some rows have a list of evidence passages; join them.
    evidence = row.get('evidence', '') or ''
    if isinstance(evidence, list):
        evidence = ' '.join(str(e) for e in evidence)
    evidence = evidence.strip()

    question = (row.get('question', '') or '').strip()
    answer   = (row.get('answer', '')   or '').strip()

    if not question or not answer:
        return None

    # Add doc context to the instruction when available
    doc_name = row.get('doc_name', '')
    doc_year = row.get('doc_year', '')
    doc_hint = f' from {doc_name} ({doc_year})' if doc_name else ''

    instruction = random.choice(template_pool) + doc_hint

    return {
        'instruction': instruction,
        'input':  f'Document excerpt:{doc_hint}\n\n{evidence}\n\nQuestion: {question}',
        'output': answer,
        'source': 'financebench',
    }

fb_examples = []
for row in tqdm(fb_dataset, desc='Converting FinanceBench'):
    ex = fb_row_to_example(row)
    if ex:
        fb_examples.append(ex)

print(f'\nFinanceBench examples: {len(fb_examples)}')

Converting FinanceBench:   0%|          | 0/150 [00:00<?, ?it/s]


FinanceBench examples: 150


## 3. Source B — SEC EDGAR (earnings transcripts + filings)

In [ ]:
# SEC EDGAR full-text search API (no key required — just a User-Agent header per SEC policy).
# We target 8-K filings (earnings calls, press releases) and 10-K annual reports.
# Strategy:
#   1. Search by form type and date range to get filing metadata
#   2. Fetch the filing index → find the primary document URL
#   3. Fetch text, extract relevant sections, synthesise instruction pairs

# !! IMPORTANT: SEC requires a descriptive User-Agent or requests will be blocked.
HEADERS = {
    'User-Agent': 'FinSight-Research your@email.com',   # ← replace with your email
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'efts.sec.gov',
}
EDGAR_SEARCH = 'https://efts.sec.gov/LATEST/search-index?q=%22earnings%22&dateRange=custom&startdt={start}&enddt={end}&forms={form}&hits.hits._source.period_of_report=true'
EDGAR_FILING = 'https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type={form}&dateb=&owner=include&count=10&search_text='
EDGAR_BASE   = 'https://www.sec.gov'

def edgar_search(form_type='8-K', start='2022-01-01', end='2024-01-01', max_results=50):
    """Search EDGAR full-text for recent filings of a given form type."""
    url = (
        f'https://efts.sec.gov/LATEST/search-index?'
        f'q=%22revenue%22+%22guidance%22&'
        f'dateRange=custom&startdt={start}&enddt={end}&'
        f'forms={form_type}&'
        f'hits.hits.total.value=true'
    )
    try:
        resp = requests.get(url, headers={**HEADERS, 'Host': 'efts.sec.gov'}, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f'  Search error: {e}')
        return {}

def fetch_filing_text(accession_no, cik):
    """Fetch the primary document text from a filing."""
    # Normalise accession number: 0001234567-23-000001 → 000123456723000001
    acc_clean = accession_no.replace('-', '')
    index_url = f'{EDGAR_BASE}/Archives/edgar/data/{cik}/{acc_clean}/{accession_no}-index.htm'
    try:
        resp = requests.get(index_url, headers={**HEADERS, 'Host': 'www.sec.gov'}, timeout=15)
        resp.raise_for_status()
        # Find the primary .htm document link in the index
        links = re.findall(r'href="(/Archives/edgar/data/[^"]+\.htm)"', resp.text, re.I)
        if not links:
            return ''
        doc_url = EDGAR_BASE + links[0]
        doc_resp = requests.get(doc_url, headers={**HEADERS, 'Host': 'www.sec.gov'}, timeout=20)
        doc_resp.raise_for_status()
        # Strip HTML tags
        text = re.sub(r'<[^>]+>', ' ', doc_resp.text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    except Exception as e:
        return ''

print('✓ EDGAR helpers defined')
print('  Tip: keep request rate ≤ 10/sec — the SEC rate-limits aggressive crawlers.')

✓ EDGAR helpers defined
  Tip: keep request rate ≤ 10/sec — the SEC rate-limits aggressive crawlers.


In [ ]:
# We'll query EDGAR's full-text search endpoint for 8-K filings (earnings releases)
# across a 2-year window and extract ~2,000 text passages.
#
# For each passage we synthesise three task types:
#   (a) Summarise — "Summarise this earnings release in plain English."
#   (b) Extract a metric — "What was the reported EPS? Cite the passage."
#   (c) Trend analysis — "Describe the revenue trend and its key drivers."

EDGAR_INSTRUCTION_TEMPLATES = {
    'summarise': [
        'Summarise the following earnings release section in plain English for a non-specialist investor.',
        'Write a brief plain-English summary of the key points in this earnings filing excerpt.',
        'Explain what the following section of a company\'s 8-K filing means for investors.',
    ],
    'extract': [
        'Extract the key financial metrics mentioned in this passage. Present them as a concise bullet list.',
        'Identify and list the specific financial figures (revenue, EPS, margins, guidance) from the excerpt below.',
        'What are the reported financial results in the following excerpt? Be specific and quantitative.',
    ],
    'trend': [
        'Based on the passage below, describe the year-over-year trends and their primary drivers.',
        'Analyse the performance trends described in this filing excerpt. What is improving and what is declining?',
        'What does the following passage reveal about the company\'s financial trajectory? Cite specific data.',
    ],
}

def passage_to_examples(passage, company='', period=''):
    """Given a cleaned text passage, create 3 instruction-following examples."""
    context = f'{company} — {period}\n\n{passage}' if company else passage
    examples = []
    for task_type, templates in EDGAR_INSTRUCTION_TEMPLATES.items():
        examples.append({
            'instruction': random.choice(templates),
            'input': context,
            'output': '[TO_GENERATE]',  # placeholder — see cell below
            'source': f'edgar_{task_type}',
            'task_type': task_type,
        })
    return examples

# --- Run a sample query to verify connectivity ---
print('Testing EDGAR search API...')
sample = edgar_search(form_type='8-K', start='2023-01-01', end='2023-06-30', max_results=5)
hits = sample.get('hits', {}).get('hits', [])
print(f'  Sample hits returned: {len(hits)}')
if hits:
    print(f'  First result: {hits[0].get("_source", {}).get("display_names", "?")}')

Testing EDGAR search API...
  Sample hits returned: 100
  First result: ['AECOM  (ACM)  (CIK 0000868857)']


In [ ]:
# ── Bulk EDGAR collection ──────────────────────────────────
# This cell fetches filing passages from EDGAR.
# We target S&P 500-level companies by searching for common financial keywords.
# Rate limit: 0.2 s sleep between requests to be polite.

# Keywords that appear in earnings filings — gives variety across sectors
EDGAR_QUERIES = [
    'revenue growth operating income',
    'earnings per share guidance outlook',
    'gross margin segment performance',
    'free cash flow capital expenditure',
    'comparable store sales same-store',
    'net interest margin loan growth',  # banking
    'combined ratio underwriting profit',  # insurance
    'recurring revenue ARR churn',  # SaaS
    'production volumes cost per barrel',  # energy
    'same-property NOI occupancy rate',  # REITs
]

edgar_passages = []
seen_hashes = set()

for query in tqdm(EDGAR_QUERIES, desc='EDGAR queries'):
    url = (
        f'https://efts.sec.gov/LATEST/search-index?'
        f'q={requests.utils.quote(query)}&'
        f'dateRange=custom&startdt=2022-01-01&enddt=2024-06-01&'
        f'forms=8-K&hits.hits._source.period_of_report=true'
    )
    try:
        resp = requests.get(url, headers={**HEADERS, 'Host': 'efts.sec.gov'}, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        hits = data.get('hits', {}).get('hits', [])
        for hit in hits[:10]:  # top 10 per query
            src = hit.get('_source', {})
            # Pull inline text highlight if available
            highlight = hit.get('highlight', {})
            passages = highlight.get('file_text', highlight.get('body_text', []))
            company  = ', '.join(src.get('display_names', ['Unknown']))
            period   = src.get('period_of_report', '')
            for p in passages[:2]:
                # Strip HTML highlight tags
                clean = re.sub(r'<[^>]+>', '', p).strip()
                if len(clean) < 100:  # skip very short snippets
                    continue
                h = hashlib.md5(clean.encode()).hexdigest()
                if h in seen_hashes:
                    continue
                seen_hashes.add(h)
                edgar_passages.append({'text': clean, 'company': company, 'period': period})
    except Exception as e:
        print(f'  Error on query "{query}": {e}')
    time.sleep(0.25)  # be polite to the SEC

print(f'\nUnique EDGAR passages collected: {len(edgar_passages)}')

EDGAR queries:   0%|          | 0/10 [00:00<?, ?it/s]


Unique EDGAR passages collected: 0


In [ ]:
# ── Synthesise outputs for EDGAR passages ─────────────────
# For summarise/extract/trend tasks, we need a ground-truth output.
# Options:
#   Option A (recommended for quality): use GPT-4o / Claude to generate outputs
#   Option B (free, lower quality): use rule-based extraction + templates
#   Option C: skip EDGAR synthesis and rely solely on FinanceBench
#
# This notebook implements Option B so it runs fully offline.
# If you have API access, replace generate_output() with an API call.

def extract_numbers(text):
    """Pull dollar amounts, percentages, and multiples from text."""
    patterns = [
        r'\$[\d,.]+\s*(?:billion|million|thousand|B|M|K)?',
        r'[\d,.]+\s*(?:billion|million|thousand)?\s*(?:dollars|USD)',
        r'[\d.]+%',
        r'\$[\d.]+\s*(?:per share|EPS)',
    ]
    found = []
    for pat in patterns:
        found.extend(re.findall(pat, text, re.I))
    return list(dict.fromkeys(found))[:8]  # deduplicate, cap at 8

def generate_output(task_type, passage, company=''):
    """Rule-based output generation (replace with LLM call for higher quality)."""
    numbers = extract_numbers(passage)
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', passage) if len(s.strip()) > 30]

    if task_type == 'extract':
        if numbers:
            return 'Key financial metrics mentioned:\n' + '\n'.join(f'- {n}' for n in numbers)
        return 'No specific financial figures were found in this excerpt.'

    elif task_type == 'summarise':
        # Take first 2 sentences as a rough summary
        summary = ' '.join(sentences[:2])
        if numbers:
            summary += f' Key figures: {", ".join(numbers[:3])}.'
        return summary or 'The excerpt describes financial performance without specific figures.'

    elif task_type == 'trend':
        trend_words = re.findall(r'\b(increase[d]?|decrease[d]?|grew|declined|rose|fell|improved|deteriorated|up|down)[^.]{0,60}\.', passage, re.I)
        if trend_words:
            return 'Trends identified:\n' + '\n'.join(f'- {t.strip()}' for t in trend_words[:4])
        return 'The excerpt does not contain explicit trend language.'

    return passage[:200]

# Build EDGAR examples
edgar_examples = []
for p in tqdm(edgar_passages, desc='Generating EDGAR examples'):
    for task_type, templates in EDGAR_INSTRUCTION_TEMPLATES.items():
        output = generate_output(task_type, p['text'], p.get('company', ''))
        if output and output != 'The excerpt does not contain explicit trend language.':
            edgar_examples.append({
                'instruction': random.choice(templates),
                'input': f"{p['company']} — {p['period']}\n\n{p['text']}" if p.get('company') else p['text'],
                'output': output,
                'source': f'edgar_{task_type}',
            })

print(f'EDGAR examples generated: {len(edgar_examples)}')

Generating EDGAR examples: 0it [00:00, ?it/s]

EDGAR examples generated: 0


## 4. Clean, deduplicate, and balance

In [ ]:
all_examples = fb_examples + edgar_examples
print(f'Total before cleaning: {len(all_examples)}')

def clean_example(ex):
    """Normalise whitespace, enforce length limits, strip residual HTML."""
    for key in ('instruction', 'input', 'output'):
        val = ex.get(key, '') or ''
        val = re.sub(r'<[^>]+>', ' ', val)   # strip HTML
        val = re.sub(r'\s+', ' ', val).strip()
        ex[key] = val
    return ex

def is_valid(ex):
    """Quality filters."""
    inst   = ex.get('instruction', '')
    inp    = ex.get('input', '')
    output = ex.get('output', '')

    # Must have all three fields
    if not inst or not inp or not output:
        return False
    # Input must have at least 50 chars of financial content
    if len(inp) < 50:
        return False
    # Output must be substantive
    if len(output) < 20:
        return False
    # Skip placeholder outputs
    if output == '[TO_GENERATE]':
        return False
    # Combined length guard (keeps examples in Mistral's context window)
    total_chars = len(inst) + len(inp) + len(output)
    if total_chars > 8000:  # ~2K tokens — safe for 8K context
        return False
    return True

# Clean and filter
cleaned = [clean_example(ex) for ex in all_examples]
filtered = [ex for ex in cleaned if is_valid(ex)]
print(f'After cleaning/filtering: {len(filtered)}')

# Deduplicate by input hash (catches near-duplicates)
seen = set()
deduped = []
for ex in filtered:
    h = hashlib.md5(ex['input'][:300].encode()).hexdigest()
    if h not in seen:
        seen.add(h)
        deduped.append(ex)

print(f'After deduplication:     {len(deduped)}')

# Source distribution
dist = defaultdict(int)
for ex in deduped:
    dist[ex['source']] += 1
print('\nSource distribution:')
for src, cnt in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {src:30s} {cnt:>5d}')

Total before cleaning: 150
After cleaning/filtering: 81
After deduplication:     75

Source distribution:
  financebench                      75


In [ ]:
# ── Train / validation split ───────────────────────────────
# 90 / 10 split, stratified by source to avoid source leakage.

from collections import defaultdict

by_source = defaultdict(list)
for ex in deduped:
    by_source[ex['source']].append(ex)

train_examples, val_examples = [], []
for src, examples in by_source.items():
    random.shuffle(examples)
    n_val = max(1, int(len(examples) * 0.10))
    val_examples.extend(examples[:n_val])
    train_examples.extend(examples[n_val:])

# Final shuffle
random.shuffle(train_examples)
random.shuffle(val_examples)

print(f'Train: {len(train_examples)}')
print(f'Val:   {len(val_examples)}')

# ── Write JSONL files ──────────────────────────────────────
def write_jsonl(examples, path):
    with open(path, 'w') as f:
        for ex in examples:
            # Keep only the three core fields for training
            record = {
                'instruction': ex['instruction'],
                'input':       ex['input'],
                'output':      ex['output'],
            }
            f.write(json.dumps(record) + '\n')
    print(f'  Wrote {len(examples)} examples → {path}')

write_jsonl(train_examples, TRAIN_FILE)
write_jsonl(val_examples,   VAL_FILE)

# Stats snapshot
stats = {
    'total': len(deduped),
    'train': len(train_examples),
    'val':   len(val_examples),
    'source_distribution': dict(dist),
    'avg_input_len':  int(sum(len(e['input'])  for e in deduped) / len(deduped)),
    'avg_output_len': int(sum(len(e['output']) for e in deduped) / len(deduped)),
}
with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2)
print(f'\nStats saved → {STATS_FILE}')
print(json.dumps(stats, indent=2))

Train: 68
Val:   7
  Wrote 68 examples → finsight_data/train.jsonl
  Wrote 7 examples → finsight_data/val.jsonl

Stats saved → finsight_data/dataset_stats.json
{
  "total": 75,
  "train": 68,
  "val": 7,
  "source_distribution": {
    "financebench": 75
  },
  "avg_input_len": 4384,
  "avg_output_len": 113
}


## 5. Inspect sample examples

In [ ]:
# Spot-check 3 random training examples
for i, ex in enumerate(random.sample(train_examples, min(3, len(train_examples)))):
    print(f'─── Example {i+1} [{ex.get("source","?")}] ───')
    print(f'INSTRUCTION: {ex["instruction"]}')
    print(f'INPUT (first 300 chars): {ex["input"][:300]}...')
    print(f'OUTPUT: {ex["output"][:300]}')
    print()

─── Example 1 [financebench] ───
INSTRUCTION: Read the excerpt from the financial filing and answer the question that follows. from Pfizer_2023Q2_10Q ()
INPUT (first 300 chars): Document excerpt: from Pfizer_2023Q2_10Q () {'evidence_text': 'We expect to incur costs of approximately $700 million in connection with separating Upjohn, of which approximately 90% has been incurred since inception\nand through the second quarter of 2023. These charges include costs and expenses r...
OUTPUT: Yes, it's spinning off Upjohn.

─── Example 2 [financebench] ───
INSTRUCTION: Answer the following question based on the provided financial document excerpt. from ADOBE_2022_10K ()
INPUT (first 300 chars): Document excerpt: from ADOBE_2022_10K () {'evidence_text': 'ADOBE INC.\nCONSOLIDATED STATEMENTS OF INCOME\n(In millions, except per share data)\n \nYears Ended\n \nDecember 2,\n2022\nDecember 3,\n2021\nNovember 27,\n2020\nRevenue:\n \nSubscription\n$ \n16,388 $ \n14,573 $ \n11,626 \nProduct\n \n532 ...


## 6. Next steps

Phase 1 is complete. You should now have:
- `finsight_data/train.jsonl` — training set
- `finsight_data/val.jsonl`   — validation set  
- `finsight_data/dataset_stats.json` — summary statistics

**Quality upgrade (optional but recommended):**
If you have access to the Anthropic or OpenAI API, replace the `generate_output()` function with an API call to generate higher-quality outputs for the EDGAR examples. Even annotating 500–1,000 examples this way can significantly lift eval scores.

**Phase 2 checklist before fine-tuning:**
- [ ] Verify train set has ≥ 3,000 examples (more is better up to ~10K)
- [ ] Spot-check 20 examples manually for output quality
- [ ] Confirm no PII or obviously harmful content
- [ ] Upload dataset to HuggingFace Hub (`datasets` library) for reproducibility
- [ ] Start Phase 2 notebook: QLoRA fine-tuning on Kaggle T4×2

In [ ]:
# Optional: push dataset to HuggingFace Hub
# from datasets import Dataset
# hf_dataset = Dataset.from_list(deduped)
# hf_dataset.push_to_hub('your-hf-username/finsight-dataset', private=False)
# print('Dataset pushed to HuggingFace Hub')